# Setup

In [1]:
!pip install xarray tiler rioxarray

Defaulting to user installation because normal site-packages is not writeable


In [2]:
# Standard library imports
import os
import subprocess
import sys
from datetime import datetime
from glob import glob
from pathlib import Path

# Third-party imports
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import numpy.ma as ma
import torch
import xarray as xr
from tqdm import tqdm
from transformers import AutoImageProcessor
from tiler import Tiler, Merger

%matplotlib inline

/usr/local/lib/python3.12/dist-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
repo_dir = "lfm"

# if not os.path.exists(repo_dir):
#     subprocess.run(["git", "clone", f"https://github.com/nasa-nccs-hpda/{repo_dir}"])
# else:
#     subprocess.run(["git", "-C", repo_dir, "pull"])

In [4]:
# sys.path.insert(0, os.path.join(os.getcwd(), repo_dir))
sys.path.insert(0, "/explore/nobackup/people/ajkerr1/Lunar_FM/latest_repo/lfm")

from lfm.tasks.sem_segmentation.sseg_model import load_dinov3_encoder, DINOSegmentation
from lfm.tasks.sem_segmentation.data_cube_inference import run_datacube_inference, plot_data_cubes

In [5]:
input_dir = "/explore/nobackup/people/ajkerr1/Lunar_FM/static_training_data_test/2026_06_05/datacubes/M115502559CE_r1950_c150_input"
print(os.listdir(input_dir))

['Cube-LTM38N_Zoom-5_Tile-4-62_ProdId-M115502559CE.tif', 'StaticCube-LTM38N_Zoom-5_Tile-4-62.tif', 'Cube-LTM38N_Zoom-5_Tile-5-62_ProdId-M115502559CE.tif', 'StaticCube-LTM38N_Zoom-5_Tile-5-62.tif', 'Cube-LTM38N_Zoom-5_Tile-4-63_ProdId-M115502559CE.tif', 'StaticCube-LTM38N_Zoom-5_Tile-4-63.tif', 'Cube-LTM38N_Zoom-5_Tile-5-63_ProdId-M115502559CE.tif', 'StaticCube-LTM38N_Zoom-5_Tile-5-63.tif']


In [8]:
from lfm.tasks.sem_segmentation.data_cube_inference import get_datacube_data
data_list, file_pairs = get_datacube_data(
    input_dir,
    max_images=None,
    verbose=True,
    verify_bands=True,
)

Searching with pattern: /explore/nobackup/people/ajkerr1/Lunar_FM/static_training_data_test/2026_06_05/datacubes/M115502559CE_r1950_c150_input/**/*.tif
Found 8 files

Found 8 total .tif files. Wac: 4, Static: 4

Processing tile_id: 4-62
  Wac: /explore/nobackup/people/ajkerr1/Lunar_FM/static_training_data_test/2026_06_05/datacubes/M115502559CE_r1950_c150_input/Cube-LTM38N_Zoom-5_Tile-4-62_ProdId-M115502559CE.tif
  Static: /explore/nobackup/people/ajkerr1/Lunar_FM/static_training_data_test/2026_06_05/datacubes/M115502559CE_r1950_c150_input/StaticCube-LTM38N_Zoom-5_Tile-4-62.tif
  Wac bands: 7

  Verifying bands for StaticCube-LTM38N_Zoom-5_Tile-4-62.tif:
  Pattern: 'lola_kaguya'
  Selected band indices (1-based): [59, 60, 61, 62, 63]
  Band verification:
    Band 59: lola_kaguya_60mpp_asp ✓
    Band 60: lola_kaguya_60mpp_cos ✓
    Band 61: lola_kaguya_60mpp_elv ✓
    Band 62: lola_kaguya_60mpp_sin ✓
    Band 63: lola_kaguya_60mpp_slp ✓
  ✓ All 5 bands match pattern
  Converting indices:

In [10]:
print(len(data_list))
for data in data_list:
    print(data.shape)

4
(12, 512, 512)
(12, 512, 512)
(12, 512, 512)
(12, 512, 512)


In [ ]:
a

# Config

In [ ]:
# Data paths
INPUT_DIR = "/explore/nobackup/projects/lfm/model_inputs/data_cubes/WAC"

# Where to load dinov3 init weights
weights_local_checkpoint = '/explore/nobackup/projects/ilab/models/dinov3/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth'
pretrained_checkpoint = "/explore/nobackup/projects/lfm/model_inference/checkpoints/sem_seg/7_band/best_model.pt"

# Output dir (this will be created automatically)
OUTPUT_DIR = "./outputs"  # Change this if you want a specific path
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

# Training hyperparameters
BATCH_SIZE = 16
NUM_WORKERS = 8

# Number of bands, band filter
NUM_BANDS = 12  # n bands to be passed through model; NOT n bands in base input!
BAND_FILTER = None  # UV bands are first 2, RGB are bands 5, 3, 2

# Model parameters
TARGET_SIZE = (304, 304)  # Input size for DINO model
FREEZE_ENCODER = False

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load model

In [ ]:
# Load model from model factory
encoder = load_dinov3_encoder(weights_local_checkpoint, device=device)
model = DINOSegmentation(
    encoder=encoder,  # Use DINOv3 head
    num_classes=2,  # Binary segmentaton (crater, not crater)
    img_size=TARGET_SIZE,
    num_bands=NUM_BANDS,
).to(device)

# Apply checkpoint
checkpoint = torch.load(pretrained_checkpoint, map_location='cpu')
checkpoint_state = checkpoint['model_state_dict']
missing_keys, unexpected_keys = model.load_state_dict(
    checkpoint_state, strict=False
)
model.to(device)
model.eval()
print("Successfully loaded model from checkpoint!")

# Load data

## Load statistics from training to normalize inputs

In [ ]:
print("\n" + "="*60)
print("STEP 1: Loading training dataset statistics.")
print("="*60)

# Load mean and std from a previous training run
MEAN = np.load("/explore/nobackup/projects/lfm/model_inference/stats/sem_seg/dataset_mean.npy")
MEAN_RGB = MEAN[BAND_FILTER]
STD = np.load("/explore/nobackup/projects/lfm/model_inference/stats/sem_seg/dataset_std.npy")
STD_RGB = STD[BAND_FILTER]
print(MEAN_RGB, STD_RGB)

# Inference

In [ ]:
# plot some data cubes
fig, axes, data, data_normalized, file_paths = plot_data_cubes(
    INPUT_DIR,
    mode="bands",
    max_images=9,
    output_path="sem_seg_inputs.png",  # Changed from output_dir
    verbose=False,
    apply_normalization=False,  # Skip normalization or compute from data
    cmap='gray',
)

In [ ]:
# Run inference, create viz of inference
n_images = 3
images, predictions = run_datacube_inference(
    model=model,
    device=device,
    input_dir=INPUT_DIR,
    mean=MEAN_RGB,
    std=STD_RGB,
    output_path="sem_seg_inference.png",
    n_images=n_images,
    model_native_size=TARGET_SIZE[0],
    tile_overlap=0.25,
    threshold=0.5,
    normalize=True,
    save_inputs_dir=None,
    use_sliding=True,
    band_filter=BAND_FILTER,
)